In [0]:
from pyspark.sql import functions as F

p = spark.table("workspace.pobreza.personas_silver")

tramo = (F.when(F.col("e27") < 6, "0-5")
          .when(F.col("e27") < 13, "6-12")
          .when(F.col("e27") < 18, "13-17")
          .when(F.col("e27") < 65, "18-64")
          .otherwise("65+"))

gold = (p.withColumn("tramo", tramo)
         .groupBy("anio", "tramo")
         .agg(F.round(F.sum(F.col("pobre17") * F.col("W_ANO")) / F.sum("W_ANO") * 100, 1)
               .alias("pct_pobreza"))
         .orderBy("anio", "tramo"))

gold.write.mode("overwrite").option("overwriteSchema", True) \
    .saveAsTable("workspace.pobreza.pobreza_por_edad_gold")

display(gold)

In [0]:
orden = (F.when(F.col("tramo") == "0-5", 1).when(F.col("tramo") == "6-12", 2)
          .when(F.col("tramo") == "13-17", 3).when(F.col("tramo") == "18-64", 4)
          .otherwise(5))
gold = gold.withColumn("orden", orden).orderBy("anio", "orden")

In [0]:
import os
from pyspark.sql import functions as F

p = spark.table("workspace.pobreza.personas_silver")

def tasa(df, *by):
    return (df.groupBy(*by).agg(
        F.round(F.sum(F.col("pobre17") * F.col("W_ANO")) / F.sum("W_ANO") * 100, 1).alias("pct_pobreza"),
        F.count("*").alias("n_muestra")))

tramo = (F.when(F.col("e27") < 6, "0-5").when(F.col("e27") < 13, "6-12")
          .when(F.col("e27") < 18, "13-17").when(F.col("e27") < 65, "18-64").otherwise("65+"))
orden = (F.when(F.col("e27") < 6, 1).when(F.col("e27") < 13, 2)
          .when(F.col("e27") < 18, 3).when(F.col("e27") < 65, 4).otherwise(5))

tablas = {
    # % de pobreza por tramo de edad
    "pobreza_por_tramo": tasa(p.withColumn("tramo", tramo).withColumn("orden", orden),
                              "anio", "orden", "tramo").orderBy("anio", "orden"),
    # % de pobreza edad por edad (para la curva)
    "pobreza_por_edad": tasa(p.filter(F.col("e27") <= 90), "anio", "e27")
                              .withColumnRenamed("e27", "edad").orderBy("anio", "edad"),
    # % de pobreza infantil (0-17) por departamento (para el mapa)
    "pobreza_infantil_dpto": tasa(p.filter(F.col("e27") < 18), "anio", "dpto", "nom_dpto")
                              .orderBy("anio", "dpto"),
}

ruta_nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
ruta_repo = os.path.dirname(ruta_nb)
if not ruta_repo.startswith("/Workspace"):
    ruta_repo = "/Workspace" + ruta_repo
carpeta = f"{ruta_repo}/data"
os.makedirs(carpeta, exist_ok=True)

for nombre, df in tablas.items():
    (df.write.mode("overwrite").option("overwriteSchema", True)
       .saveAsTable(f"workspace.pobreza.{nombre}_gold"))
    df.toPandas().to_json(f"{carpeta}/{nombre}.json", orient="records", force_ascii=False, indent=1)
    print("OK:", nombre)

print("JSON guardados en:", carpeta)
display(tablas["pobreza_infantil_dpto"])